In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geopandas as gpd
import xarray as xr
import rioxarray as rxr
from shapely import box
import rasterio
from rasterio.mask import mask

import pystac_client
import planetary_computer

from IPython.display import Image

sys.path.append("../utils")

In [2]:
inspections = gpd.read_file(
    "/capstone/wildfire_prep/data/PUZZLE_PIECES/inspections_master_training_geometries.geojson"
).drop(columns = 'apn')
inspections.head()


,inspection_id,Date,year,month,status,geometry
0,1,2019-06-05,2019,6,Compliant,"POLYGON ((-2892.357 -371905.217, -2813.179 -37..."
1,2,2019-05-09,2019,5,Compliant,"POLYGON ((-8475.394 -370860.259, -8368.416 -37..."
2,3,2019-05-09,2019,5,Compliant,"POLYGON ((-8253.618 -371183.727, -8136.129 -37..."
3,4,2019-05-09,2019,5,Compliant,"POLYGON ((-8201.866 -370989.432, -8091.086 -37..."
4,5,2019-12-17,2019,12,Compliant,"POLYGON ((-8065.883 -370931.624, -7966.144 -37..."


In [3]:
# Reduce size of dataset, for testing
# inspections = inspections.head(20000)

In [4]:
# Initial Test Cell

# inspections = inspections.to_crs(epsg=4326)

# catalog = pystac_client.Client.open(
#     "https://planetarycomputer.microsoft.com/api/stac/v1",
#     modifier=planetary_computer.sign_inplace,
# )

# row = inspections.iloc[0]
# geom = row.geometry

# # 3) Compute the bbox
# minx, miny, maxx, maxy = geom.bounds

# # 3) Search June 2021 over that bbox
# search = catalog.search(
#     collections=["sentinel-2-l2a"],
#     bbox=[minx, miny, maxx, maxy],
#     datetime="2021-06-01/2021-06-30",
#     query={"eo:cloud_cover": {"lte": 5}},
# )


# items = search.item_collection()
# print(f"Returned {len(items)} Items")
# items

# item = items[0]

# with rasterio.open(item.assets["B04"].href) as src:
#     raster_crs = src.crs

# # now reproject your inspections into the raster’s CRS
# inspections = inspections.to_crs(raster_crs)
# single_row = single_row.to_crs(raster_crs)

# red_href = item.assets["B04"].href
# nir_href = item.assets["B08"].href

# with rasterio.open(red_href) as src_red, rasterio.open(nir_href) as src_nir:
#     red_clip, red_transform = mask(src_red, single_row.geometry, crop=True)
#     nir_clip, nir_transform = mask(src_nir, single_row.geometry, crop=True)


# # Convert to float32 to avoid integer division
# red = red_clip.astype("float32")
# nir = nir_clip.astype("float32")

# # Compute NDVI = (NIR − Red) / (NIR + Red)
# ndvi = (nir - red) / (nir + red)

# print(f"NDVI array shape:", ndvi.shape)

# mean_ndvi = np.nanmean(ndvi)
# mean_ndvi

# single_row.loc[row.name, "mean_ndvi"] = mean_ndvi
# single_row

# print(f"Mean NDVI for inspection {row.name}: {mean_ndvi:.4f}")

In [5]:
# First Iteration of Function

# import calendar
# import pystac_client
# import planetary_computer
# import rasterio
# from rasterio.mask import mask
# import numpy as np


# def add_mean_ndvi(dataframe):
#     # work on a copy of the user’s original
#     df_out = dataframe.copy()
#     df_query = df_out.to_crs(epsg=4326)

#     catalog = pystac_client.Client.open(
#         "https://planetarycomputer.microsoft.com/api/stac/v1",
#         modifier=planetary_computer.sign_inplace,
#     )

#     mean_vals = []

#     for idx, row in df_query.iterrows():
#         geom = row.geometry

#         # build month window
#         year, month = int(row["year"]), int(row["month"])
#         last_day = calendar.monthrange(year, month)[1]
#         start = f"{year}-{month:02d}-01"
#         end = f"{year}-{month:02d}-{last_day:02d}"

#         # search
#         minx, miny, maxx, maxy = geom.bounds
#         search = catalog.search(
#             collections=["sentinel-2-l2a"],
#             bbox=[minx, miny, maxx, maxy],
#             datetime=f"{start}/{end}",
#             query={"eo:cloud_cover": {"lte": 5}},
#         )
#         items = search.item_collection()

#         if not items:
#             # no scenes found → record NaN and move on
#             mean_vals.append(np.nan)
#             continue

#         item = items[0]

#         # get COG CRS
#         with rasterio.open(item.assets["B04"].href) as src:
#             cog_crs = src.crs

#         # reproject this one geometry into the COG CRS
#         single = df_query.loc[[idx]].to_crs(cog_crs)
#         mask_geom = [single.geometry.iloc[0]]

#         # clip & compute NDVI
#         with (
#             rasterio.open(item.assets["B04"].href) as src_red,
#             rasterio.open(item.assets["B08"].href) as src_nir,
#         ):
#             red_clip, _ = mask(src_red, mask_geom, crop=True)
#             nir_clip, _ = mask(src_nir, mask_geom, crop=True)

#         red = red_clip.astype("float32")
#         nir = nir_clip.astype("float32")
#         with np.errstate(divide="ignore", invalid="ignore"):
#             ndvi = (nir - red) / (nir + red)

#         mean_vals.append(float(np.nanmean(ndvi)))

#     # attach back to the original dataframe
#     df_out["mean_ndvi"] = mean_vals
#     return df_out


In [6]:
# Second Iteration of Function

# import calendar
# import pystac_client
# import planetary_computer
# import rasterio
# from rasterio.mask import mask
# import numpy as np


# def add_mean_ndvi(dataframe):

#     # Copy the user's original dataframe to avoid messing up their original
#     df_out = dataframe.copy()
#     df_query = df_out.to_crs(epsg=4326)

#     # Open the catalog
#     catalog = pystac_client.Client.open(
#         "https://planetarycomputer.microsoft.com/api/stac/v1",
#         modifier=planetary_computer.sign_inplace,
#     )

#     # Declare an empty list for the mean ndvi values
#     mean_vals = []

#     # Iterate over the dataframe by inspection_id
#     for insp_id in df_query["inspection_id"]:
#         # pull the row by its inspection_id
#         row = df_query[df_query["inspection_id"] == insp_id].iloc[0]
#         geom = row.geometry

#         # Build date range based on dataframe 'year' and 'month' columns
#         year, month = int(row["year"]), int(row["month"])
#         last_day = calendar.monthrange(year, month)[1]
#         start = f"{year}-{month:02d}-01"
#         end = f"{year}-{month:02d}-{last_day:02d}"

#         # Search for overlapping scenes, by inspection geometry
#         try:
#             search = catalog.search(
#                 collections=["sentinel-2-l2a"],
#                 bbox=geom.bounds,
#                 datetime=f"{start}/{end}",
#                 query={"eo:cloud_cover": {"lte": 5}},
#             )
#             item = next(search.items(), None)

#         except APIError as e:
#             print(f"inspection_id {insp_id}: STAC APIError, skipping → NaN")
#             mean_vals.append(np.nan)
#             # optional backoff
#             time.sleep(0.5)
#             continue

#         # If no scenes found, make this NaN and continue
#         if not item:
#             mean_vals.append(np.nan)
#             print(f"inspection_id {insp_id}: no scene found, NDVI set to NaN")
#             continue
        

#         # Get the scene's CRS to match the geometry to
#         with rasterio.open(item.assets["B04"].href) as src:
#             cog_crs = src.crs

#         # Reproject the geometry to that CRS
#         single = df_query[df_query["inspection_id"] == insp_id].to_crs(cog_crs)
#         mask_geom = [single.geometry.iloc[0]]

#         try:
#             with (
#                 rasterio.open(item.assets["B04"].href) as src_red,
#                 rasterio.open(item.assets["B08"].href) as src_nir,
#             ):
#                 red_clip, _ = mask(src_red, mask_geom, crop=True)
#                 nir_clip, _ = mask(src_nir, mask_geom, crop=True)

#         except ValueError as e:
#             # this happens if shapes really don't overlap
#             mean_vals.append(np.nan)
#             print(f"inspection_id {insp_id}: geometry did not overlap, NDVI set to NaN")
#             continue

#         red = red_clip.astype("float32")
#         nir = nir_clip.astype("float32")
        
#         with np.errstate(divide="ignore", invalid="ignore"):
#             ndvi = (nir - red) / (nir + red)

#         mean_val = float(np.nanmean(ndvi))
#         mean_vals.append(mean_val)
#         print(f"inspection_id {insp_id}: NDVI calculated = {mean_val:.4f}")

#     # attach back to the original dataframe in the same order
#     df_out["mean_ndvi"] = mean_vals
#     return df_out


In [7]:
# Third Iteration of Function

import time
import calendar
import numpy as np
import rasterio
from rasterio.mask import mask

import pystac_client
from pystac_client.stac_api_io import APIError
import planetary_computer


def add_mean_ndvi(dataframe):

    df_out = dataframe.copy()

    df_query = df_out.to_crs(epsg=4326)

    # Open the catalog once
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )

    mean_vals = []

    for insp_id in df_query["inspection_id"]:
        row = df_query[df_query["inspection_id"] == insp_id].iloc[0]
        geom = row.geometry

        # Build date range
        year, month = int(row["year"]), int(row["month"])
        last_day = calendar.monthrange(year, month)[1]
        start = f"{year}-{month:02d}-01"
        end = f"{year}-{month:02d}-{last_day:02d}"

        # Search but at NaN if API Error is found
        # Search built from month and year columns in dataframe
        # Searching for imagery with <5% cloud cover
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=geom.bounds,
                datetime=f"{start}/{end}",
                query={"eo:cloud_cover": {"lte": 5}},
            )
            item = next(search.items(), None)

        except APIError:
            print(f"inspection_id {insp_id}: STAC API Timeout Error: skipping, NaN")
            mean_vals.append(np.nan)
            time.sleep(0.5)
            continue

        if item is None:
            print(f"inspection_id {insp_id}: no scene found, NaN")
            mean_vals.append(np.nan)
            continue

        # Get CRS of raster
        with rasterio.open(item.assets["B04"].href) as src:
            cog_crs = src.crs

        # Reproject geometry
        single = df_query[df_query["inspection_id"] == insp_id].to_crs(cog_crs)
        mask_geom = [single.geometry.iloc[0]]

        # Clip and compute NDVI
        try:
            with (
                rasterio.open(item.assets["B04"].href) as src_red,
                rasterio.open(item.assets["B08"].href) as src_nir,
            ):
                red_clip, _ = mask(src_red, mask_geom, crop=True)
                nir_clip, _ = mask(src_nir, mask_geom, crop=True)
        except ValueError:
            print(f"inspection_id {insp_id}: geometry did not overlap, NaN")
            mean_vals.append(np.nan)
            continue
        
        # Calculate NDVI, supressing warnings.
        red = red_clip.astype("float32")
        nir = nir_clip.astype("float32")
        with np.errstate(divide="ignore", invalid="ignore"):
            ndvi = (nir - red) / (nir + red)

        mean_val = float(np.nanmean(ndvi))
        mean_vals.append(mean_val)
        print(f"inspection_id {insp_id}: NDVI calculated = {mean_val:.4f}")

    df_out["mean_ndvi"] = mean_vals
    return df_out


In [8]:
inspections = add_mean_ndvi(inspections)
print(inspections.loc[:, ["inspection_id", "mean_ndvi"]])


inspection_id 1: no scene found, NaN
inspection_id 2: NDVI calculated = 0.4539
inspection_id 3: NDVI calculated = 0.3516
inspection_id 4: NDVI calculated = 0.2804
inspection_id 5: NDVI calculated = 0.5746
inspection_id 6: NDVI calculated = 0.3786
inspection_id 7: NDVI calculated = 0.6879
inspection_id 8: NDVI calculated = 0.3984
inspection_id 9: NDVI calculated = 0.3438
inspection_id 10: NDVI calculated = 0.5298
inspection_id 11: NDVI calculated = 0.7154
inspection_id 12: NDVI calculated = 0.6442
inspection_id 13: NDVI calculated = 0.5906
inspection_id 14: NDVI calculated = 0.6300
inspection_id 15: NDVI calculated = 0.4136
inspection_id 16: NDVI calculated = 0.5543
inspection_id 17: NDVI calculated = 0.4139
inspection_id 18: NDVI calculated = 0.6007
inspection_id 19: NDVI calculated = 0.6358
inspection_id 20: NDVI calculated = 0.3675
inspection_id 21: NDVI calculated = 0.6208
inspection_id 22: NDVI calculated = 0.2661
inspection_id 23: NDVI calculated = 0.7018
inspection_id 24: NDVI ca

KeyboardInterrupt: 

In [ ]:
n_nans = inspections["mean_ndvi"].isna().sum()
print(f"There are {n_nans} missing (NaN) values.")


In [ ]:
print(f"There are {inspections["mean_ndvi"].isna().sum()} missing (NaN) values.")
